# Mini WM: a tour of quick world models for robots

This notebook is a **summary** of the models in this folder. The code lives in the Python files, not here: each section
says what a model predicts, how it learns and how it acts, and names the functions to read next.

| Section | File | Paper |
|---|---|---|
| 1. mini-FM, the policy every other model builds on | [mini_fm.py](mini_fm.py) | Octo ([arXiv 2405.12213](https://arxiv.org/abs/2405.12213)), π0's action expert |
| 2. mini-Dreamer, a pixel world model | [mini_dreamer.py](mini_dreamer.py) | DreamerV3, Dreamer 4 ([arXiv 2509.24527](https://arxiv.org/abs/2509.24527)) |
| 3. mini-TD-MPC2, a world model that plans | [mini_tdmpc2.py](mini_tdmpc2.py) | TD-MPC2 ([arXiv 2310.16828](https://arxiv.org/abs/2310.16828)) |
| 4. mini-WAM, a world action model | [mini_wam.py](mini_wam.py) | Cosmos Policy ([arXiv 2601.16163](https://arxiv.org/abs/2601.16163)) |
| 5. mini-JEPA, a latent world model | [mini_jepa.py](mini_jepa.py) | RoboJEPA ([arXiv 2610.10515](https://arxiv.org/abs/2610.10515)) |
| 6. mini-VLAPS, search with a world model and a policy | [mini_vlaps.py](mini_vlaps.py) | VLAPS ([arXiv 2508.12211](https://arxiv.org/abs/2508.12211)) |
| 7. How the world models differ | | |
| 8. A trained mini-FM in the LIBERO-90 sim | [sim_eval.py](sim_eval.py) | |
| 9. RL fine-tuning with GRPO or PPO | [mini_fm_rl.py](mini_fm_rl.py) | GRPO, PPO |
| The training loop they all share | [mini_train.py](mini_train.py) | |

Everything runs on **LIBERO-90**, both the data (`gberseth/libero_90_no_noops_lerobot`) and the sim: the default
configs of this repo (`conf/dataset/default.yaml`, `conf/sim/default.yaml`).

![What each model predicts](docs/policy_vs_world_models.svg)

**How to use it:** run the setup cell once. The data section (0) runs top to bottom. Each model section after it
depends only on the setup cell, so you can run those in any order, or run just one. A model cell builds the model from
its real config (`conf/`), runs one forward pass on a real LIBERO-90 batch and prints the model's size and its
(untrained) losses. That shows what goes in and what comes out. Training is done from the command line (the last
section). The diagrams are in the style of the Robot Learning book.


## Setup

Run this cell first. Every model section below depends only on it. The two helpers are:
- `config(name)` loads `conf/<name>.yaml` with hydra, as `mini_train.py` does, for LIBERO-90 data and sim;
- `libero90_batch(cfg)` returns a real training batch from a `RobotCache` of a few LIBERO-90 demonstrations. It also
  fills in the values `mini_train.py` computes from the data: the vocabulary, the action and pose sizes, and the
  whitening stats.

The data is the Hugging Face dataset [gberseth/libero_90_no_noops_lerobot](https://huggingface.co/datasets/gberseth/libero_90_no_noops_lerobot)
(`LIBERO_90`). To watch its demonstrations in the browser, open it in the
[LeRobot dataset visualizer](https://huggingface.co/spaces/lerobot/visualize_dataset?path=/gberseth/libero_90_no_noops_lerobot/episode_0).


In [ ]:
import os, sys, numpy as np, torch
import matplotlib.pyplot as plt
from IPython import display
from PIL import Image
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf
os.environ.setdefault("MUJOCO_GL", "egl")  ## LIBERO renders off-screen
HERE = os.path.abspath("")  ## the repo, this notebook's folder
OUT = os.path.join(HERE, "outputs", "mini_wm_tutorial")  ## what the notebook writes: GIFs, sim videos
os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, HERE)
import mini_fm, mini_wam, mini_dreamer, mini_jepa, mini_tdmpc2
from data_loading import load_lerobots, stream_lerobot_episode, build_vocab, RobotCache, to_device, POSE_KEY

LIBERO_90 = "gberseth/libero_90_no_noops_lerobot"

def config(name):
    with initialize_config_dir(config_dir=os.path.join(HERE, "conf"), version_base=None):
        cfg = compose(config_name=name)
    OmegaConf.set_struct(cfg, False)
    cfg.device = "cpu"
    return cfg

def libero90_batch(cfg, n_episodes=8, batch_size=16):
    cfg.dataset.robot_image_key = [list(cfg.dataset.cameras) if cfg.dataset.cameras else cfg.dataset.robot_image_key]
    frames, episodes, tasks = load_lerobots([LIBERO_90], cfg.dataset.robot_image_key)
    episodes = episodes.sample(n_episodes, random_state=cfg.r_seed).reset_index(drop=True)  ## a few, so they decode in seconds
    rows = np.concatenate([np.arange(s, s + n) for s, n in zip(episodes["dataset_from_index"], episodes["length"])])
    data = frames.select(rows).with_format("numpy")
    actions, poses = np.stack(data["action"]), np.stack(data[POSE_KEY])
    ## The whitening stats, here of these episodes only (mini_train.py uses every train episode)
    stats = lambda x: {"mean": x.mean(0).tolist(), "std": np.maximum(x.std(0), 1e-6).tolist()}
    cfg.dataset.action_stats = {LIBERO_90: stats(actions)}
    cfg.dataset.action_mean, cfg.dataset.action_std = stats(actions).values()
    cfg.dataset.pose_mean, cfg.dataset.pose_std = stats(poses).values()
    cfg.action_dim, cfg.pose_dim = actions.shape[1], poses.shape[1]
    chars, encode_txt, _ = build_vocab(list(cfg.dataset.chars_list))
    cfg.dataset.chars_list, cfg.vocab_size, cfg.action_bins = chars, len(chars), len(chars)
    ## The world models also get the frames (and actions, and Dreamer and TD-MPC2 the rewards) after the next F actions
    m = cfg.model
    future = {"wam": cfg.policy.action_stacking, "dreamer": m.sequence_steps, "jepa": m.sequence_steps,
              "tdmpc2": m.horizon * cfg.policy.action_stacking}.get(m.type, 0)
    cache = RobotCache(cfg, encode_txt, frames, episodes, tasks, "cpu", n_episodes, {LIBERO_90: 1.0}, future_steps=future,
                       future_actions=m.type in ("dreamer", "jepa", "tdmpc2"), future_rewards=m.type in ("dreamer", "tdmpc2"))
    k = cfg.dataset.robot_frames_per_episode  ## frames from each sampled episode
    return to_device(cache.sample(batch_size // k, k), cfg)

def summary(model, parts):
    print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")
    print("losses:", {name: round(value.item(), 4) for name, value in parts.items()})

def show_batch(batch):
    """ The shape of each part of a batch: mini-FM's 5, then the world models' future frames, actions and rewards. """
    names = ["X_text", "X_images", "targets", "actions", "pose", "frames", "sequence_actions", "rewards"]
    for name, x in zip(names, batch):
        print(f"{name:>16}: {tuple(x.shape)}")

## 0. The data: LIBERO-90

[LIBERO-90](https://libero-project.github.io/) has 90 short tasks for a Franka Panda in kitchen, living-room and study
scenes: 3,933 teleoperated demonstrations, recorded at 20 Hz, with 73 distinct goal texts (some tasks share one). [openvla](https://github.com/openvla/openvla)'s scripts re-render them (without
the no-op actions) into the LeRobot v3 format on the Hub: the frames are in parquet files, each camera's images in MP4
files. Training's `load_lerobots` downloads the whole dataset (about 600 MB) and reads it without the lerobot library.
To look at the data, `stream_lerobot_episode` reads just one demonstration from the Hub, in a few seconds. It returns
- `frames`: one row per timestep of that demonstration (the action, the joint positions, the episode and the task);
- `video`: its camera images, side by side;
- `episodes`: one row per demonstration of the dataset (its length and where its videos are);
- `tasks`: the goal text of each task index.

The cells of this section run top to bottom.


In [ ]:
CAMERAS = ["observation.images.wrist_image", "observation.images.image"]  ## wrist, then agent view
frames, video, episodes, tasks = stream_lerobot_episode(LIBERO_90, CAMERAS, index=0, size=(128, 128))
print(f"{len(episodes)} episodes, {episodes['length'].sum()} frames, {len(tasks)} tasks")
print(f"episode length (steps at 20 Hz): mean {episodes['length'].mean():.0f}, min {episodes['length'].min()}, max {episodes['length'].max()}")
print("frame columns:", list(frames.columns))
print("some goals:", *tasks[:5], sep="\n  ")

One demonstration, with both cameras side by side (wrist on the left, agent view on the right). The model sees each
frame at 64x64 pixels per camera.


In [ ]:
print("goal:", tasks[frames["task_index"].iloc[0]], f"({len(video)} steps)")  ## video: (length, 128, 256, 3)
gif = os.path.join(OUT, "episode.gif")
Image.fromarray(video[0]).save(gif, save_all=True, append_images=[Image.fromarray(f) for f in video[1::4]], duration=200, loop=0)
display.display(display.Image(gif))

**Actions and pose.** Each action has 7 dimensions: the end effector's motion (delta x, y, z), its rotation
(axis-angle) and the gripper (-1 open, +1 closed). Every dataset in this repo uses this format. The model's
input pose (with `policy.use_pose_data`) is the arm's 7 joint positions. It is deliberately in a different space from
the actions, so the model can't read the next action off its own pose.

The models predict an **action chunk**: the action at a frame and the next 4 (`policy.action_stacking` = 5), whitened
with the dataset's mean and std. The sim runs all 5 before it queries the model again.


In [ ]:
actions, joints = np.stack(frames["action"]), np.stack(frames[POSE_KEY])
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for i, name in enumerate(["dx", "dy", "dz", "rx", "ry", "rz", "gripper"]):
    axes[0].plot(actions[:, i], label=name)
axes[0].set(title="actions", xlabel="step (20 Hz)"); axes[0].legend(ncol=4, fontsize=8)
axes[1].plot(joints); axes[1].set(title="joint positions (the pose input)", xlabel="step (20 Hz)")
plt.show()


**The goal text** is tokenized one character per token (`build_vocab`) and cut to `max_block_size` (16) characters.
Many LIBERO-90 goals begin with the same 16 characters, so the model sees them as the same goal and must tell them apart
from the image. `max_block_size=64` keeps more of each goal.


In [ ]:
chars, encode_txt, decode_txt = build_vocab(list(config("mini-fm").dataset.chars_list))
print(f"vocabulary: {len(chars)} characters")
print("encoded:", encode_txt(tasks[0][:16]), "->", repr(decode_txt(encode_txt(tasks[0][:16]))))
for n in (16, 32, 64):
    print(f"{n} characters: {len({t[:n] for t in tasks})} distinct goals out of {len(tasks)}")


**A training batch.** In training, a `RobotCache` picks episodes (weighted by each dataset's share of the mix), then
`dataset.robot_frames_per_episode` (4) frames in each, all with tensor ops on the cached trajectories. This is the
batch every model below starts from:

![One training sample from a LIBERO-90 demonstration](docs/libero_batch.svg)

| | shape | what it holds |
|---|---|---|
| `X_text` | (B, 16) | the goal text |
| `X_images` | (B, 64, 64 * cameras, 3) | the camera frame(s), pixels in [-1, 1] |
| `targets` | (B,) | -1: a robot row, scored on its actions |
| `actions` | (B, 5 * 7) | the action chunk, whitened |
| `pose` | (B, 7) | the joint positions, whitened |

This is a behaviour-cloning batch: the inputs at a frame and the action chunk that follows. The world models
(sections 2 to 5) train on the same batch with one camera, plus the frames that follow, and each section shows what
its batch adds.

In [ ]:
cfg = config("mini-fm")  ## two cameras side by side
show_batch(libero90_batch(cfg))


## 1. mini-FM: the policy

[mini_fm.py](mini_fm.py) is a small Octo-style transformer, built from [micro-vlm](https://github.com/milarobotlearningcourse/micro-vlm). Given the goal
text and a camera frame, it predicts an **action chunk**: the next `policy.action_stacking` (5) actions, which the sim
runs before it asks the model again. It has no model of the world. Every other model in this folder either contains it
or is compared against it.

**The sequence** (`FM.forward`): image patch tokens (`get_patches_fast`, 8x8 patches; with two cameras side by side, 128
tokens), then the goal text (16 character tokens), then one class token. The mask (`MultiHeadAttentionFast`) lets the
image tokens see every token, while the text and the class token attend causally.

![mini-FM's token sequence and attention mask](docs/fm_sequence.svg)

**The action head** reads the class token (`model.action_head`):
- `huber` (the default), `mse` or `l1` (`MSEActionHead`): a linear layer regresses the whitened chunk. RL fine-tuning
  treats this prediction as the mean of a Gaussian (`MSEActionHead.log_prob`).
- `diffusion` (`DiffusionActionHead`): flow matching with an *action expert*, as π0. The expert is a second, small
  transformer over the 5 noisy action tokens. At every layer it attends to the backbone's keys and values. It can
  represent a chunk with several modes (going left or right around an object), where regression averages them.

![Regression and flow-matching action heads](docs/fm_action_heads.svg)

**Acting:** `FM.act` returns the whitened chunk. `FMPolicy` wraps the model for the sim evals: it encodes the goal
text, flips the sim's image to match the training data and un-whitens the actions with the stats of the dataset of the
task being run.


In [ ]:
cfg = config("mini-fm")  ## two cameras side by side, wrist and agent view
batch = libero90_batch(cfg)
model = mini_fm.make_model(cfg)
logits, loss, parts = model(*batch)
summary(model, parts)
print("act ->", tuple(model.act(*batch[:2], pose=None).shape), "(B, chunk * action_dim)")


**From mini-FM to a VLA.** mini-FM is a vision-language-action model in miniature: images and a goal text in,
actions out. A full VLA such as OpenVLA ([Kim et al. 2024, arXiv 2406.09246](https://arxiv.org/abs/2406.09246),
[github.com/openvla/openvla](https://github.com/openvla/openvla)) keeps the same idea, but with every part scaled up:

![OpenVLA: two vision encoders and Llama 2 7B, which writes the actions as tokens](docs/openvla.svg)

| | mini-FM | OpenVLA |
|---|---|---|
| Backbone | a ~3M-parameter transformer, trained from scratch | Llama 2 7B, a language model pre-trained on web text |
| Vision | 8x8 pixel patches, a linear layer | two pre-trained encoders, DINOv2 (where things are) and SigLIP (what they are), their features projected into the language model's tokens |
| Language | one token per character, 16 characters | Llama's subword tokenizer, the whole instruction |
| Actions | a head on the class token regresses (or samples) a 5-action chunk | each of the 7 dimensions cut into 256 bins, written as 256 rarely used words of Llama's vocabulary, predicted one token at a time like text |
| Data | LIBERO-90 | about 970k robot episodes from the Open X-Embodiment mix, many robots |

The step from one to the other is mostly about **where the knowledge comes from**. mini-FM learns what a "black bowl"
looks like only from the demonstrations. OpenVLA starts from a vision-language model (Prismatic) that already
learned, from internet images and text, to name objects and follow instructions. Robot data then only has to teach it
how to move, and it carries some of that web knowledge to objects and phrasings no demonstration contains.

Two design choices separate VLAs, and mini-FM's options sit on the same axes:
- **Actions as tokens or as continuous values.** OpenVLA predicts discrete action tokens with the language model's own
  loss, so nothing in the architecture changes. Its follow-up OpenVLA-OFT
  ([arXiv 2502.19645](https://arxiv.org/abs/2502.19645)) replaced them with a regression head that predicts a whole
  chunk at once (L1 loss, as `model.action_head=l1`), which is faster and more accurate on LIBERO. π0 uses a
  flow-matching action expert (`diffusion` here).
- **Adapting it.** A 7B model is fine-tuned on a new robot with low-rank adapters (LoRA), or post-trained with RL
  (section 9 does that for mini-FM).

![One transformer with text, images, video and actions as inputs and outputs](docs/multimodal_fm.svg)

**One model, all kinds of data.** Once the policy is a language model reading tokens, any data that can be written
as tokens can train it. PaLM-E ([Driess et al. 2023, arXiv 2303.03378](https://arxiv.org/abs/2303.03378),
[palm-e.github.io](https://palm-e.github.io/)) made this point at scale: it put a ViT-22B vision encoder into the
540B-parameter PaLM language model (562B in all). Images, robot states and scene representations are encoded into the
same embedding space as words, and interleaved with text as "multimodal sentences". One model is trained on all of it
together: internet-scale visual question answering and captioning, plus planning data from several robots (tabletop
pushing, task-and-motion planning, mobile manipulation in a kitchen). It answers in text, e.g. the next step of a
plan, which a separate low-level policy carries out. Two findings matter here:
- **Transfer across data:** training on the mix did better on each robot's tasks than training on that robot's data
  alone, with far fewer robot examples. The web data teaches what objects are and how instructions are phrased; the
  robot data only has to ground them.
- **Little forgetting at scale:** the largest model kept most of its language abilities, and set a state of the art
  on the OK-VQA benchmark at the time.

VLAs such as RT-2 and OpenVLA take the next step: the same model also writes the low-level actions as tokens. mini-FM
has the hooks for this kind of co-training, at a tiny scale: it shares micro-VLM's token head and accepts text-only
rows (a blank image), but today it trains on robot rows only.

The LIBERO datasets used here were built with a copy of OpenVLA's data processing code.


## 2. mini-Dreamer: a forward model in pixels

[mini_dreamer.py](mini_dreamer.py) is DreamerV3's world model, `p(s' | s, a)`, as a network of its own:
`Dreamer` holds a mini-FM (`self.fm`), trained next to it, and mini-FM is the policy the sim
evals run: it is Dreamer's actor. Like Dreamer's actor, it predicts one action at a time
(`policy.action_stacking` = 1 in `conf/mini-dreamer.yaml`), and the sim queries it at every step. Its network is `model.policy_net`: `fm`, mini-FM's transformer (the default), or `small_convnet` (`SmallConvNetFM`: three convolutions, the cheapest to run in imagination). It keeps DreamerV3's reward head and critic: the demonstrations get TD-MPC2's
sparse reward (1 after an action from which the demonstration has reached its last frame), and the critic learns the
value of their states.

**The RSSM** (`rssm_step`): a CNN encodes each frame. A GRU steps a deterministic state `h` from the last latent `z` and
the action. The *prior* predicts `z` from `h` alone, and the *posterior* from `h` and the encoded frame. A decoder
rebuilds the frame from `(h, z)`. `model.latent` is `categorical` (DreamerV3: 32 one-hot variables of 32 classes, with
1% uniform mixed in) or `gaussian` (DreamerV1).

![mini-Dreamer's RSSM, unrolled](docs/dreamer_rssm.svg)

**Loss** (`world_model_loss`, over the current frame and the frames after the next `model.sequence_steps` = 16 actions):
- `frames`: the squared error summed over each frame's pixels (the log-likelihood);
- `kl_dyn = KL(sg(posterior) || prior)`, which trains the prior to predict what the posterior will see;
- `kl_rep = KL(posterior || sg(prior))`, weighted less, which keeps the posterior predictable;
- `reward`: the reward after each action, predicted from the state `(h, z)` it leads to (squared error in symlog space);
- `critic` (`critic_loss`): the value of each state, regressed in symlog space towards DreamerV3's lambda-return of the
  real rewards, bootstrapped with a slow-moving copy of the critic (`update_target`, `model.critic_tau`). It learns on
  the posterior's states of the real sequence. DreamerV3 trains it on imagined rollouts of its actor; here the actor
  is mini-FM, so the values are those of the demonstrations' actions.

The reward and critic losses train the RSSM too, as the frame loss does, so the state `(h, z)` holds what predicts the
rewards and values, not only the pixels.

Both KLs use free bits. **Imagination** encodes only the first frame. After that, the prior rolls forward on the
actions alone: `frame_grid` logs it on the demonstrations' actions.

**mini-FM learns in imagination** (`actor_loss`), as Dreamer's actor, following the actor loss of DreamerV2
([arXiv 2010.02193](https://arxiv.org/abs/2010.02193), section "Actor loss"), with its REINFORCE gradients. From the
state of each real frame, mini-FM acts on the decoded frame, sampling its action from a Gaussian of fixed std
(`model.actor_std`) around its prediction, and the prior steps on that action, for `model.imagine_horizon` (16) steps.
The world model only generates and scores these trajectories, without gradients: the reward head scores each step
and the critic each state. The return from step `t` is DreamerV2's λ-target, a mixture of the predicted rewards and
the critic's values up to `H` = 16, then the critic's value of the last state from there on:

`V_t = r_t+1 + γ ((1 - λ) v_t+1 + λ V_t+1)` for `t < H`, and `V_H = v_H` (`model.return_lambda` = 0.95), where
`r_t+1` is the reward of action `a_t`, predicted at the state it leads to.

Each action's log-probability is weighted by its advantage, `V_t - v_t` (the critic's value is the baseline):

`actor = -mean(log π(a_t | ô_t) · (V_t - v_t))`

It is weighted by `model.actor_coef`, next to mini-FM's imitation loss, which keeps it close to the demonstrations.


**Its training batch** is mini-FM's (section 0) with one camera, plus the frame after each of the next 16 actions
and those actions, and the reward after each action. The RSSM steps once per action.

![mini-Dreamer's training batch](docs/batch_dreamer.svg)

| | shape | what it holds |
|---|---|---|
| `frames` | (B, 64, 64, 16 * 3) | the frame after each of the next 16 actions; predicted at 32 x 32 |
| `sequence_actions` | (B, 16 * 7) | those 16 actions, whitened |
| `rewards` | (B, 16) | 1 after an action from which the last frame is reached, as TD-MPC2's |
| `actions` | (B, 1 * 7) | mini-FM's target: a chunk of 1 action |

In [ ]:
cfg = config("mini-dreamer")
batch = libero90_batch(cfg)
show_batch(batch)
model = mini_dreamer.make_model(cfg)
logits, loss, parts = model(*batch)
summary(model, parts)


## 3. mini-TD-MPC2: a world model that plans its actions

[mini_tdmpc2.py](mini_tdmpc2.py) is TD-MPC2 at its 5M size, learned offline from the demonstrations. It is the only
model here that **is its own policy**: there is no mini-FM, and every action comes from planning in its latent space.

![mini-TD-MPC2: the latent model and planning with MPPI](docs/tdmpc2.svg)

- **Model:** a CNN encodes the frame into a latent `z` (bounded by `SimNorm`). MLPs predict the next `z` (`next`), the
  reward (`reward`), the return (`Qs`, an ensemble of 5) and the policy prior (`pi`). All are conditioned on a task
  embedding read from the goal text.
- **A step is an action chunk** (5 actions), and the reward is 1 once the demonstration has reached its end.
- **Losses** (`TDMPC2.loss`, over `model.horizon` = 3 chunks): `consistency` (the predicted latents against the encoded
  real frames), then `reward_ce` and `value_ce` (cross-entropy over two-hot symlog bins, Q towards the TD target
  `r + γ Q_target(z', π(z'))`), then `pi_loss` (maximize Q plus a little entropy). `update_target` moves the target
  Q-functions.
- **Acting** (`plan`, MPPI): sample 512 action sequences, some from `pi` and the rest from a Gaussian. Score each one
  with its predicted rewards plus Q at the end (`estimate_value`), refit the Gaussian to the 64 best, repeat 6 times,
  then run the first chunk. With `model.mpc=false` it takes `pi`'s mean instead.


**Its training batch** is mini-FM's (section 0) with one camera, plus the next 3 chunks (15 actions), the frames
after them and a reward for each. Each step of the model is one chunk, so it uses the frame and the reward after each
chunk's last action. The reward is 1 after an action from which the demonstration has reached its last frame, the goal;
past the end of a demonstration its last frame and action repeat.

![TD-MPC2's training batch](docs/batch_tdmpc2.svg)

| | shape | what it holds |
|---|---|---|
| `frames` | (B, 64, 64, 15 * 3) | the frame after each of the next 15 actions |
| `sequence_actions` | (B, 15 * 7) | those actions: 3 chunks of 5 |
| `rewards` | (B, 15) | 1 after an action from which the last frame is reached |

In [ ]:
cfg = config("mini-tdmpc2")
batch = libero90_batch(cfg)
show_batch(batch)
model = mini_tdmpc2.make_model(cfg)
_, loss, parts = model(*batch)
summary(model, parts)
print("planned chunk ->", tuple(model.act(*batch[:2], pose=None).shape))


## 4. mini-WAM: a world action model

[mini_wam.py](mini_wam.py) is mini-FM's transformer, trained to predict **the next 5 frames as well as the actions**,
after Cosmos Policy. The idea: if one network must imagine how the scene will change, the features its actions are
read from should carry more information about the world.

**The sequence** (`WAM.features`): mini-FM's tokens, then the patches of the 5 future frames (at `model.frame_size`, 32
px), each with noise added for a random diffusion step. The same blocks learn to remove the noise (DDPM with a cosine
schedule, `cosine_noise_schedule`). They predict the clean patches, not the noise: at the noisiest steps a noisy patch is nearly all noise, so
predicting the noise is easy but says nothing about the frames.

![mini-WAM's sequence and mask](docs/wam_sequence.svg)

**The mask is the key trick** (`WAM.__init__`). The class token, and every token before it, cannot see the frame
tokens. Acting therefore runs only mini-FM's part of the sequence and costs the same as mini-FM. Predicting the future
shapes the shared weights, but never feeds into an action at test time.

**Loss:** `mse_head_weights * action loss + frame_loss_weight * frames`. `predict_frames` samples the future, and
`frame_grid` draws the real and predicted frames side by side (logged as `predicted_frames`).


**Its training batch** is mini-FM's (section 0) with one camera, plus the frame after each action of the chunk.
The chunk is already in the batch (`actions`), so the frames and the actions it predicts are one sequence.

![mini-WAM's training batch](docs/batch_wam.svg)

| | shape | what it holds |
|---|---|---|
| `frames` | (B, 64, 64, 5 * 3) | the frame after each of the chunk's 5 actions, stacked in the channels; predicted at 32 x 32 |

In [ ]:
cfg = config("mini-wam")
batch = libero90_batch(cfg)
show_batch(batch)
model = mini_wam.make_model(cfg)
logits, loss, parts = model(*batch)
summary(model, parts)


## 5. mini-JEPA: a forward model in latent space

[mini_jepa.py](mini_jepa.py) follows RoboJEPA, the latest of the JEPAs (joint-embedding predictive architectures). Like
mini-Dreamer, it predicts the future after the next 16 actions. The difference: it **never predicts pixels**, only the
*embeddings* of future frames. The recipe comes from I-JEPA, which learns features from still images, so we start there.

**Where it comes from: I-JEPA** ([Assran et al. 2023, arXiv 2301.08243](https://arxiv.org/abs/2301.08243)). I-JEPA
learns an image encoder without labels. It hides a few blocks of an image and learns to predict them from the rest. It
does not predict their pixels, only the embeddings an encoder gives them. Three networks are trained together:

![I-JEPA: the three networks, and what is hidden for images, video and robots](docs/ijepa.svg)

- **Context encoder** (a ViT, trained by gradients): reads only the visible *context block*, one large block (85–100%
  of the image) with the target blocks' patches removed. It gives one embedding per visible patch.
- **Target encoder** (no gradients): a copy of the context encoder whose weights follow it as an exponential moving
  average (EMA), momentum 0.996 rising to 1. It reads the *whole* image. The targets are cut from its output: the
  embeddings of the patches in M = 4 *target blocks*, each 15–20% of the image.
- **Predictor** (a narrow ViT, trained by gradients): reads the context embeddings plus one *mask token* per target
  patch, a shared learned vector plus the position of the patch it stands for. It predicts the embeddings of the
  target patches. The loss is the squared L2 distance to the target encoder's embeddings, averaged over the M blocks.

Why each part is needed:
- **The target encoder:** without it, the targets would come from the context encoder itself, with gradients through
  both. The easiest way to reach zero loss would then be an encoder that maps every image to the same embedding
  (*collapse*). A slow EMA copy is a moving target the encoder cannot shift to meet its predictions, and in practice
  the embeddings stay informative.
- **The predictor:** it keeps the encoder's job general. The encoder only has to describe what it sees; the predictor
  does the guessing about what is hidden, and is thrown away after training. Only the target encoder is kept.
- **Embeddings, not pixels:** a pixel target (as in MAE) spends capacity on texture and lighting that can't be guessed
  from context anyway. Predicting embeddings lets the model skip those details and keep what can be inferred: which
  object is there, and where. In the paper's ablation, the same model reaches 66.9% ImageNet linear-probe accuracy
  predicting target-encoder embeddings, and 40.7% predicting pixels. It also needs no augmented views of the image, as
  the invariance methods (DINO, iBOT) do, and it converges in about 5× fewer iterations than MAE.
- **The masks:** the target blocks must be large enough to be about objects, not single patches, and the context large
  and spread out enough to infer them.

**From images to video to robots.** The three networks stay the same; only *what is hidden* changes:
- **Video, V-JEPA** ([Bardes et al. 2024, arXiv 2404.08471](https://arxiv.org/abs/2404.08471)): the patches are
  space-time tubes of a short clip, and the same blocks are hidden in every frame. The predictor can't copy a hidden
  patch from the frame before, so it has to learn how things move. V-JEPA uses an L1 loss.
- **Robots, V-JEPA 2-AC** ([Assran et al. 2025, arXiv 2506.09985](https://arxiv.org/abs/2506.09985)) and RoboJEPA:
  what is hidden is *the future*. The encoder, pre-trained on internet video as V-JEPA, is frozen. An
  action-conditioned predictor is trained on robot videos with their actions: from the current frame's embedding and
  the next actions, predict the next frames' embeddings. That predictor is a world model. A robot plans with it by
  searching for the actions whose predicted embedding lands nearest a goal image's.

mini-JEPA is that last step, at a small scale:

| I-JEPA | mini-JEPA |
|---|---|
| context encoder, on the context block | `encoder`, on the current frame (and, teacher-forced, the real frames before each step) |
| target encoder, the EMA copy | `target_encoder`, moved by `update_target` |
| predictor, with mask tokens at the hidden patches' positions | `JEPA.predict`: the next frame's patch tokens, with the action token of the 4 actions that lead there |
| the target blocks of the same image | the next frames (one every 4 actions) |
| squared L2 | L1, as V-JEPA and RoboJEPA |

RoboJEPA's encoder is a frozen V-JEPA 2.1 (1B parameters). There is no small pre-trained one, so mini-JEPA's encoder
learns together with the predictor, with I-JEPA's EMA target encoder to keep it from collapsing.

**mini-JEPA's parts:**

![mini-JEPA: encoder, moving-average targets and predictor](docs/jepa.svg)

- **Encoder** (`Encoder`): a small ViT that gives one embedding per patch of a frame.
- **Predictor** (`JEPA.predict`): a transformer whose timesteps are one action token (`model.frame_skip` = 4 actions)
  plus the frame's patch tokens. Attention is timestep-causal, and it predicts the next frame's embeddings.
- **Loss** (`world_model_loss`): ½ teacher-forced L1 (`latent_tf`: each step starts from the real frames) plus
  ½ autoregressive L1 (`latent_ar`: a `rollout` from the first frame only, the imagination error).
- **Collapse:** the loss would reach 0 if every embedding were the same. The targets therefore come from an
  exponential moving average of the encoder (`target_encoder`, moved by `update_target` after each optimizer step), as
  in I-JEPA. Watch `target_std`: it falls towards 0 if the embeddings collapse. `latent_copy` is the error of
  predicting "nothing changes", a baseline the model should beat.

**Acting** (`model.policy`):
- `cem` (the default, as RoboJEPA, `JEPA.plan`): the cross-entropy method searches for the actions that bring the
  scene to a goal image. It samples 256 sequences of 16 actions, imagines each with `rollout`, scores it by the L1
  distance between the last imagined embedding and the goal image's, refits a Gaussian to the 32 best, repeats 5 times,
  then runs the first chunk. As in RoboJEPA, the goal text is never used. The sim gives no goal image, so each eval
  episode starts from the init state of one of the dataset's demonstrations and its goal is that demo's last frame
  (`sim_eval.dataset_goal_images`).
- `fm`: a mini-FM trained next to the world model, as in mini-Dreamer, is the policy.


**Its training batch** is mini-FM's (section 0) with one camera, plus the frame after each of the next 16 actions
and those actions, as mini-Dreamer's. It steps at 5 Hz (`model.frame_skip` = 4): every 4th frame is a target (its
patch embeddings), and each group of 4 actions is one action token.

![mini-JEPA's training batch](docs/batch_jepa.svg)

| | shape | what it holds |
|---|---|---|
| `frames` | (B, 64, 64, 16 * 3) | the frame after each of the next 16 actions; every 4th is a target |
| `sequence_actions` | (B, 16 * 7) | those 16 actions, 4 action tokens of 4 |

In [ ]:
cfg = config("mini-jepa")
batch = libero90_batch(cfg)
show_batch(batch)
model = mini_jepa.make_model(cfg)
logits, loss, parts = model(*batch)
summary(model, parts)


## 6. mini-VLAPS: searching with a world model and a policy

[mini_vlaps.py](mini_vlaps.py) follows VLAPS. It **trains nothing**: it takes a pre-trained mini-Dreamer checkpoint,
whose RSSM is the world model and whose mini-FM (trained next to it, section 2) is the policy, and searches before each
chunk the sim runs.

![mini-VLAPS: the tree search in the world model, after VLAPS's Figure 1](docs/vlaps.svg)

The search is Monte Carlo tree search over action chunks (`VLAPS.search`):
- **Select:** from the root, take the child with the highest `prior * sqrt(N(node) + 1) / (1 + N(child))`. This is
  PUCT without its value term, as in VLAPS: only the policy's prior and the visit counts steer the search.
- **Expand** (`expand`): the node's candidates are the policy's chunk at the node's imagined frame and noisy copies of
  it. Their prior is a softmax of the negative distance to the policy's chunk, mixed with uniform. The world model steps
  through each one to a child.
- **Roll out:** every new child runs the policy in the world model for a few chunks. The policy reads the frames the
  RSSM decodes.
- **Reward:** sparse, as in VLAPS. It counts as a success once an imagined frame is close to the goal image, the last
  frame of the demonstration that started from the same init state. VLAPS uses the simulator's own success instead.

The first chunk of the path to an imagined success runs; with none, the root's most visited chunk runs. The search then
starts again from the next real frame. `vlaps.simulations=0` runs mini-FM alone, the baseline the search should beat.
A run needs a trained mini-Dreamer and the sim, so there is no cell here:
`uv run mini-wm/mini_train.py --config-name mini-vlaps init_model=<path to miniDreamer_best.pth>`.


## 7. How the world models differ

| | What the world model predicts | Where it predicts | Who acts | How the world model helps the policy |
|---|---|---|---|---|
| mini-FM | nothing (actions only) | — | itself | — |
| mini-Dreamer | the frames after the next 16 actions, rewards and values | pixels, through a latent RSSM | a mini-FM trained next to it, one action per step | directly: mini-FM also maximizes the returns it imagines in the world model |
| mini-TD-MPC2 | latents, rewards and returns | latent only, no decoder | itself, by MPPI planning | directly: every action is planned with it |
| mini-WAM | the next 5 frames, by diffusion | pixels, in the policy's own transformer | its mini-FM part | shared features (the frame tokens are masked from the actions) |
| mini-JEPA | the embeddings of the frames after the next 16 actions | latent only, no decoder | itself, by CEM planning towards a goal image (or a mini-FM trained next to it) | directly, with CEM: every action is planned with it |

Two axes to compare along:
- **Pixels or latents.** Pixel models (WAM, Dreamer) can be checked by eye (`predicted_frames`), but they spend capacity
  on details that don't matter for control. Latent models (TD-MPC2, JEPA) avoid that, but they need another guard
  against collapse: TD-MPC2's reward and value losses, and JEPA's moving-average targets.
- **What the model is for:** better features (WAM), imagination (Dreamer) or planning (TD-MPC2 towards rewards, JEPA towards a goal image).


## 8. A trained mini-FM in the LIBERO-90 sim

The cells above use untrained models. This one loads a mini-FM pre-trained on LIBERO-90 (`checkpoints/miniFM_libero90.pth`:
25k steps on both cameras; the model RL fine-tuning starts from) and runs it closed loop in LIBERO-90, the same way the sim evals
in training do (`eval_sim_suites` in [sim_eval.py](sim_eval.py)). `FMPolicy` turns each sim observation into the model's
inputs, and the sim runs each predicted chunk of 5 actions before it asks again. LIBERO-90 has 90 tasks; this runs 3
of them, one episode each, which takes a few minutes on the CPU. The episode videos go to
`outputs/mini_wm_tutorial/sim_eval/libero_90/`.


In [ ]:
import glob, mini_fm_rl
from sim_eval import eval_sim_suites
CHECKPOINT = os.path.join(HERE, "checkpoints", "miniFM_libero90.pth")
model, cfg = mini_fm_rl.load_pretrained(CHECKPOINT)  ## the model and the config it was trained with
cfg.device = "cpu"
cfg.sim.simulators.libero.tasks_per_eval = 3  ## of LIBERO-90's 90 tasks, one episode each
cwd = os.getcwd()
os.chdir(OUT)  ## eval_sim_suites writes its videos to ./sim_eval/
try:
    results = eval_sim_suites(mini_fm.FMPolicy(model.eval(), cfg), cfg, iter=0)["libero_90"]
finally:
    os.chdir(cwd)
print(f"success rate {results['success_rate']:.2f} over {results['num_episodes']} episodes; per task:",
      {task: r["success_rate"] for task, r in results["per_task"].items()})
display.display(display.Video(sorted(glob.glob(os.path.join(OUT, "sim_eval", "libero_90", "*.mp4")))[0], embed=True, width=400))


## 9. RL fine-tuning: mini_fm_rl.py

[mini_fm_rl.py](mini_fm_rl.py) takes a pre-trained mini-FM (`miniFM_best.pth`, with a regression head) and improves it
on success in LIBERO-90, in the sim of section 8. A run takes hours, so there is no cell here.

![RL fine-tuning: rollouts, advantages, clipped update](docs/rl_finetune.svg)

- **The policy** is a Gaussian around the predicted chunk, with a fixed std (`action_std`). Its log-probability is the
  scaled squared error, `MSEActionHead.log_prob`.
- **Rollouts:** `rollout_libero` (in [sim_eval.py](sim_eval.py)), on a pool of LIBERO envs that stays open for the whole run.
  Each iteration runs groups of `group_size` rollouts, each group from one init state.
- **Advantage:** GRPO takes each rollout's success minus its group's mean, divided by the group's std (no value
  network). PPO uses GAE (`gae`) from a reward of 1 on success, with a value head on the policy's detached features
  (`make_value_head`).
- **Update** (`policy_update`): PPO's clipped objective per action value. The clip is asymmetric (`clip_high` >
  `clip_low`, as DAPO). Each episode is weighted equally, and queries with zero advantage are skipped.


## The shared training loop: mini_train.py

[mini_train.py](mini_train.py) trains all of them. `model.type` picks the file (`MODELS`); each file has a
`make_model(cfg)` and a `make_policy(cfg)`, and every `forward` returns `(logits, loss, parts)`. The steps of
`my_main`:

![The shared pre-training loop](docs/train_loop.svg)

1. **Data:** `load_lerobots` reads the LeRobot datasets (parquet files plus MP4s). It splits train and val by episode,
   and computes each dataset's action stats for whitening (they are saved in the config, so sim evals can un-whiten).
   `RobotCache` decodes every frame once and keeps the trajectories in RAM or on the GPU. Its `sample` also returns the
   future frames, actions and rewards the world models need.
2. **Augmentation:** `augment` shifts each image randomly (`random_shift`, as DrQ) and can jitter its colours. Future
   frames get the same shift as their current frame.
3. **Speed:** the whole forward is one `torch.compile` graph with CUDA graphs. For this reason `forward` has no
   data-dependent branches and reads its config from `plain_config`.
4. **Step:** AdamW with a linear LR decay, in bf16 autocast. After each optimizer step, `update_target` (TD-MPC2, JEPA)
   moves the target networks.
5. **Evals:** `estimate_loss` (train/val per head), `estimate_dataset_loss` (`val_by_dataset/`), `frame_grid` images for
   the world models, and sim evals in a background process (`BackgroundSimEval`). The model with the best sim
   success is saved as `<model>_best.pth`.

## Running them for real

From the repo root, on LIBERO-90 (the default configs):
```
MUJOCO_GL=egl uv run mini_train.py                            # mini-FM
MUJOCO_GL=egl uv run mini_train.py --config-name mini-wam     # mini-WAM
MUJOCO_GL=egl uv run mini_train.py --config-name mini-dreamer # mini-Dreamer
MUJOCO_GL=egl uv run mini_train.py --config-name mini-jepa    # mini-JEPA
MUJOCO_GL=egl uv run mini_train.py --config-name mini-tdmpc2  # mini-TD-MPC2
MUJOCO_GL=egl uv run mini_train.py --config-name mini-fm-rl   # RL fine-tuning of checkpoints/miniFM_libero90.pth
MUJOCO_GL=egl uv run mini_train.py --config-name mini-vlaps init_model=<run folder>/miniDreamer_best.pth  # mini-VLAPS
```
Add `max_iters=500 sim.eval_interval=499` for a quick check. The smoke tests in [tests/](tests/) run each model for a few
hundred steps. See [README.md](README.md) for Slurm and Weights & Biases.
